# Stage 3 Veri Analizi: 16 Sınıflı Model Fizibilite Raporu

Bu notebook, 32 diş ve 4 hastalık türünden oluşan veri setimizi, önerilen **16 Sınıflı Yapı** (4 Diş Tipi x 4 Hastalık) için analiz eder.

**Hedef:** Hangi (Diş Tipi + Hastalık) kombinasyonunda kaç adet gerçek veri olduğunu görmek.

**Sınıf Yapısı:**
- **Diş Grupları:** Incisor (Kesici), Canine (Köpek), Premolar (Küçük Azı), Molar (Büyük Azı)
- **Hastalıklar:** Impacted, Caries, Deep Caries, Periapical Lesion
- **Toplam Sınıf:** 4 x 4 = 16 Sınıf

In [ ]:
import json
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- CONFIG ---
TRAIN_JSON = "../data/raw/train/training_data/quadrant-enumeration-disease/train_quadrant_enumeration_disease.json"
VAL_JSON = "../data/raw/validation_triple.json"

# Mappings
DISEASE_MAP = {
    0: "Impacted",
    1: "Caries",
    2: "Periapical_Lesion",
    3: "Deep_Caries"
}

TOOTH_GROUP_MAP = {
    0: "Incisor",   # 1, 2
    1: "Canine",    # 3
    2: "Premolar",  # 4, 5
    3: "Molar"      # 6, 7, 8
}

def get_tooth_group(fdi_num):
    """
    Convert FDI number (e.g., 18, 24) to Tooth Group ID.
    Using the user's logic:
    tooth_num = fdi % 10 (1-8)
    But user provided logic assumes 'cat2_id' is FDI? Or relative? 
    Let's stick to standard Dentex FDI:
    1, 2 -> Incisor
    3 -> Canine
    4, 5 -> Premolar
    6, 7, 8 -> Molar
    """
    if fdi_num is None: return None
    
    # FDI numbers are 11-18, 21-28, 31-38, 41-48.
    # Modulo 10 gives 1-8.
    tooth_num = fdi_num % 10
    
    if tooth_num in [1, 2]:
        return 0 # Incisor
    elif tooth_num == 3:
        return 1 # Canine
    elif tooth_num in [4, 5]:
        return 2 # Premolar
    elif tooth_num in [6, 7, 8]:
        return 3 # Molar
    else:
        return None

def process_json(json_path, split_name):
    if not os.path.exists(json_path):
        print(f"File not found: {json_path}")
        return []

    with open(json_path, 'r') as f:
        data = json.load(f)
        
    results = []
    
    # We iterate annotations.
    # We need category_id_2 (Tooth Number) and category_id_3 (Disease)
    # Standard Dentex JSON usually has multiple category_ids.
    # Assuming:
    # category_id_1: Quadrant
    # category_id_2: Tooth Enum (FDI-like or raw enum? usually FDI)
    # category_id_3: Disease
    
    for ann in data['annotations']:
        cat2 = ann.get('category_id_2') # Tooth Number
        cat3 = ann.get('category_id_3') # Disease
        
        # If no disease info, it is likely healthy, but user asked for Disease Classes Analysis.
        # We check if cat3 is valid disease.
        if cat3 not in DISEASE_MAP:
            continue
            
        disease_name = DISEASE_MAP[cat3]
        
        # Determine Tooth Group
        # The `get_tooth_class` user provided used `int(cat2_id) % 10`. 
        # Assuming category_id_2 is the FDI number direct (e.g. 11, 48)
        # If it is missing, we skip.
        if cat2 is None:
            continue
            
        group_id = get_tooth_group(cat2)
        if group_id is None: continue
        
        group_name = TOOTH_GROUP_MAP[group_id]
        
        combined_class = f"{group_name}_{disease_name}"
        
        results.append({
            "split": split_name,
            "tooth_num": cat2,
            "group": group_name,
            "disease": disease_name,
            "class": combined_class
        })
        
    return results

print("Analyzing Data...")
train_data = process_json(TRAIN_JSON, "Train")
val_data = process_json(VAL_JSON, "Val")

all_data = train_data + val_data
df = pd.DataFrame(all_data)

if df.empty:
    print("No disease data found / parsing failed.")
else:
    print(f"Total Pathological Teeth Found: {len(df)}")
    print(df.head())


In [ ]:
# --- 1. PIVOT TABLE (TRAIN) ---
if not df.empty:
    train_df = df[df['split'] == 'Train']
    pivot_train = train_df.pivot_table(index='group', columns='disease', values='class', aggfunc='count', fill_value=0)
    
    # Reorder columns/index for better view
    row_order = ['Incisor', 'Canine', 'Premolar', 'Molar']
    col_order = ['Caries', 'Deep_Caries', 'Impacted', 'Periapical_Lesion']
    
    # Filter existing cols only
    row_order = [r for r in row_order if r in pivot_train.index]
    col_order = [c for c in col_order if c in pivot_train.columns]
    
    pivot_train = pivot_train.loc[row_order, col_order]
    
    print("\n--- TRAIN DATASET COUNT MATRIX ---")
    display(pivot_train)
    
    # Heatmap of Train
    plt.figure(figsize=(10, 6))
    sns.heatmap(pivot_train, annot=True, fmt="d", cmap="YlGnBu", cbar=False)
    plt.title("Train Data Distribution (Counts)")
    plt.show()

In [ ]:
# --- 2. DETAILED LIST (TRAIN + VAL) ---
if not df.empty:
    counts = df.groupby(['split', 'class']).size().reset_index(name='count')
    counts = counts.sort_values(by=['split', 'count'], ascending=[False, False])

    print("\n--- DETAILED CLASS COUNTS ---")
    # Format as a table
    print(f"{'Split':<10} {'Class Name':<30} {'Count':<5}")
    print("-"*50)
    for _, row in counts.iterrows():
        print(f"{row['split']:<10} {row['class']:<30} {row['count']:<5}")

In [ ]:
# --- 3. FEASIBILITY CHECK ---
if not df.empty:
    print("\n--- FEASIBILITY ANALYSIS ---")
    train_counts = df[df['split'] == 'Train'].groupby('class').size()
    
    critical_classes = []
    good_classes = []
    
    for cls, count in train_counts.items():
        if count < 50:
            critical_classes.append((cls, count))
        else:
            good_classes.append((cls, count))
            
    print("✅ VIABLE CLASSES (>50 samples):")
    for c in good_classes: print(f"  - {c[0]}: {c[1]}")
    
    print("\n⚠️ CRITICAL CLASSES (<50 samples) - Risk of Overfitting:")
    for c in critical_classes: print(f"  - {c[0]}: {c[1]}")